# Improvement 1 Evaluation Visualizations

This notebook rebuilds the report figures from the Improvement 1 result tables. It does not rerun retrieval or reranking and does not modify experimental inputs. Run from the repository root or from this notebook's directory. Charts are saved in `improvement/improvement_1/figures/`.

The historical TEST outputs have an internal consistency audit, alongside unresolved source/manifest hash differences. Reproduced plots describe the saved results and do not resolve that provenance limitation.


## 1. Setup and configuration

Paths are resolved relative to the repository root. Matplotlib is used directly for consistent, deterministic figures.

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, Path(__file__).resolve().parents[2] if '__file__' in globals() else here]
    for candidate in candidates:
        if (candidate / 'improvement' / 'improvement_1').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing improvement/improvement_1.')

ROOT = find_repo_root()
BASE = ROOT / 'improvement' / 'improvement_1'
FIGURES = BASE / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 220, 'font.size': 9,
                     'axes.titlesize': 11, 'axes.labelsize': 9,
                     'legend.fontsize': 8, 'figure.facecolor': 'white'})
SEED = 20261112
print(f'Repository root: {ROOT}')
print(f'Figure output: {FIGURES}')


Repository root: D:\NAM 4\pooladapt
Figure output: D:\NAM 4\pooladapt\improvement\improvement_1\figures


## 2. Load and validate data

The aggregate tables are cross-checked against the query-level handoff, paired statistical results, failure cases, and descriptive query-group table. A missing or malformed required input stops execution with a useful error.

In [3]:
def read_csv(relative: str, required: set[str]) -> pd.DataFrame:
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f'Required artifact not found: {path}')
    frame = pd.read_csv(path)
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    if frame.empty:
        raise ValueError(f'{path.name} contains no rows')
    return frame

summary = read_csv('improvement/improvement_1/tables/main_results_summary.csv',
    {'method','operating_point','n_queries','ndcg_at_10','recall_at_10','mrr_at_10','average_actual_pairs'})
query_rows = read_csv('improvement/improvement_1/matched_budget/query_level_evaluation.csv',
    {'query_id','method','operating_point','actual_reranked_pairs','ndcg_at_10','recall_at_10','mrr_at_10','reranking_latency_ms'})
latency = read_csv('improvement/improvement_1/latency_cost/latency_cost_summary.csv',
    {'method','operating_point','n_queries','mean_ms','p50_ms','p95_ms'})
pareto = read_csv('improvement/improvement_1/pareto/pareto_analysis.csv',
    {'method','operating_point','pareto_objective','objective_x_value','objective_y_value','pareto_efficient'})
paired = read_csv('improvement/improvement_1/statistics/paired_statistical_analysis.csv',
    {'operating_point','metric','n_queries','mean_paired_difference','bootstrap_95_ci_low','bootstrap_95_ci_high'})
failures = read_csv('improvement/improvement_1/failure_analysis/failure_case_analysis.csv',
    {'query_id','failure_category','operating_point'})
groups = read_csv('improvement/improvement_1/query_group_analysis/evaluation/subgroup_comparison.csv',
    {'grouping_strategy','group_name','query_count','target_budget','metric','PoolAdapt_mean','Fixed_Prefix_mean'})

metric_cols = ['ndcg_at_10','recall_at_10','mrr_at_10']
if summary.duplicated(['method','operating_point']).any():
    raise ValueError('Duplicate method/operating-point rows in main summary.')
if not summary[metric_cols].apply(lambda c: c.between(0, 1).all()).all():
    raise ValueError('A retrieval metric is outside its expected [0, 1] range.')
if query_rows['query_id'].nunique() != 300:
    raise ValueError(f"Expected 300 query IDs, found {query_rows['query_id'].nunique()}.")
if query_rows.duplicated(['query_id','method','operating_point','seed']).any():
    raise ValueError('Duplicate query/method/operating-point/seed records detected.')
if (query_rows['actual_reranked_pairs'] < 0).any() or (query_rows['reranking_latency_ms'] < 0).any():
    raise ValueError('Negative pair count or latency detected.')
if paired['n_queries'].nunique() != 1 or int(paired['n_queries'].iloc[0]) != 300:
    raise ValueError('Paired statistical rows do not consistently use 300 queries.')

print(f'Aggregate rows: {len(summary)}; query-level rows: {len(query_rows)}; unique TEST queries: {query_rows.query_id.nunique()}')
print(f'Paired comparisons: {len(paired)}; failure records: {len(failures)}; subgroup rows: {len(groups)}')
print('Budget convention: target budget is a label; actual reranked pairs are read independently.')


Aggregate rows: 21; query-level rows: 11100; unique TEST queries: 300
Paired comparisons: 9; failure records: 151; subgroup rows: 80
Budget convention: target budget is a label; actual reranked pairs are read independently.


## Shared plotting helpers

The helper functions standardize labels, save high-resolution PNGs, and retain a machine-readable inventory of generated outputs.

In [ ]:
from IPython.display import display

generated = []
COLORS = {'PoolAdapt':'#2463a6', 'Fixed Prefix':'#c44e52', 'SAGE-SLO':'#238b70',
          'PACE-EF':'#7a52a1', 'Random':'#dd8a00', 'Full Rerank':'#333333'}

def save_figure(fig, filename: str) -> None:
    path = FIGURES / filename
    fig.savefig(path, bbox_inches='tight', facecolor='white')
    display(fig) 
    plt.close(fig)
    generated.append(path)
    print(f'Saved {path.relative_to(ROOT)}')

def budget_number(value: str) -> int:
    return int(str(value).lstrip('B'))

def selected(frame: pd.DataFrame, methods: list[str], budgets: list[int]) -> pd.DataFrame:
    out = frame[frame.method.isin(methods)].copy()
    out['budget'] = out.operating_point.map(budget_number)
    return out[out.budget.isin(budgets)].sort_values('budget')


## 3. Matched-budget evaluation

The first figure shows quality at all saved target budgets. The second uses actual reranked pairs, so it does not treat a target-budget label as equal realized workload.

In [5]:
quality = selected(summary, ['PoolAdapt','Fixed Prefix'], [10,20,30,50])
fig, axes = plt.subplots(1, 3, figsize=(6.2, 3.3), sharex=True)
for ax, metric, label in zip(axes, metric_cols, ['nDCG@10','Recall@10','MRR@10']):
    for method in ['PoolAdapt','Fixed Prefix']:
        part = quality[quality.method == method]
        ax.plot(part.budget, part[metric], marker='o', linewidth=2, label=method, color=COLORS[method])
    ax.set(title=label, xlabel='Target budget (pairs)', ylim=(0.60, 0.86), xticks=[10,20,30,50])
    ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Mean query-level score')
axes[-1].legend(frameon=False, loc='lower right')
fig.suptitle('PoolAdapt and Fixed Prefix quality across target budgets', y=1.03)
fig.tight_layout()
save_figure(fig, 'report_matched_budget_quality.png')

fig, ax = plt.subplots(figsize=(6.2, 4.0))
for method in ['PoolAdapt','Fixed Prefix']:
    part = quality[quality.method == method]
    ax.plot(part.budget, part.average_actual_pairs, marker='o', linewidth=2,
            label=method, color=COLORS[method])
ax.plot([10,20,30,50],[10,20,30,50], linestyle=':', color='#888888', label='Target budget')
ax.set(xlabel='Target budget (pairs)', ylabel='Mean actual reranked pairs / query',
       title='Target budget and realized reranking workload', xticks=[10,20,30,50])
ax.grid(alpha=.25); ax.legend(frameon=False)
fig.tight_layout(); save_figure(fig, 'report_actual_reranked_pairs.png')


Saved improvement\improvement_1\figures\report_matched_budget_quality.png
Saved improvement\improvement_1\figures\report_actual_reranked_pairs.png


## 4. Latency and inference cost

Latency is reranking-stage latency (pair preparation, Cross-Encoder inference, and sorting); retrieval, selection, loading, and warm-up are excluded. It is not end-to-end latency or a monetary/GPU-cost estimate.

In [6]:
lat = selected(latency, ['PoolAdapt','Fixed Prefix'], [10,20,30,50])
fig, ax = plt.subplots(figsize=(6.2, 4.2))
width = 3.2
for offset, method in [(-width/2,'PoolAdapt'),(width/2,'Fixed Prefix')]:
    part = lat[lat.method == method]
    x = part.budget.to_numpy() + offset
    mean = part.mean_ms.to_numpy()
    p95 = part.p95_ms.to_numpy()
    ax.bar(x, mean, width=width, color=COLORS[method], alpha=.82, label=f'{method} mean')
    ax.scatter(x, p95, marker='_', s=220, linewidths=2, color='#222222', zorder=3,
               label='P95' if method == 'PoolAdapt' else None)
ax.set(xlabel='Target budget (pairs)', ylabel='Reranking-stage latency (ms)',
       title='Mean latency and P95 by target operating point', xticks=[10,20,30,50])
ax.grid(axis='y', alpha=.25); ax.legend(frameon=False)
fig.tight_layout(); save_figure(fig, 'report_reranking_latency.png')


Saved improvement\improvement_1\figures\report_reranking_latency.png


## 5. Pareto analysis

The Pareto plot uses the saved quality-versus-actual-pairs analysis across all methods and operating points. A point is efficient only if no other saved configuration has both no greater workload and no lower nDCG@10, with at least one strict improvement.

In [7]:
front = pareto[pareto.pareto_objective == 'quality_vs_actual_pairs'].copy()
front['efficient'] = front.pareto_efficient.astype(str).str.lower().eq('true')
fig, ax = plt.subplots(figsize=(6.2, 4.5))
for method, part in front.groupby('method'):
    ax.scatter(part.objective_x_value, part.objective_y_value,
               c=COLORS.get(method,'#777777'), marker='o', s=65,
               facecolors=None, label=method, alpha=.9)
    for _, row in part.iterrows():
        if row.efficient:
            ax.annotate(f"{row.method} {row.operating_point}",
                        (row.objective_x_value,row.objective_y_value), xytext=(4,4),
                        textcoords='offset points', fontsize=7)
ax.set(xlabel='Mean actual reranked pairs / query', ylabel='Mean nDCG@10',
       title='Quality versus actual reranking workload')
ax.grid(alpha=.25); ax.legend(frameon=False, ncol=2)
fig.tight_layout(); save_figure(fig, 'report_quality_cost_pareto.png')


Saved improvement\improvement_1\figures\report_quality_cost_pareto.png


## 6. Statistical analysis

The interval plot displays paired PoolAdapt-minus-Fixed-Prefix differences and saved query-bootstrap 95% intervals for the primary B10/B20/B50 comparisons. The repository also reports sign-flip permutation tests with Holm correction by metric family; this plot alone is not a significance test.

In [8]:
plot_stats = paired.copy()
plot_stats['budget'] = plot_stats.operating_point.map(budget_number)
plot_stats['metric'] = pd.Categorical(plot_stats.metric, ['nDCG@10','Recall@10','MRR@10'], ordered=True)
plot_stats = plot_stats.sort_values(['metric','budget'])
labels = [f'{m}, B{b}' for m,b in zip(plot_stats.metric.astype(str),plot_stats.budget)]
y = np.arange(len(plot_stats))
mid = plot_stats.mean_paired_difference.to_numpy()
lo = plot_stats.bootstrap_95_ci_low.to_numpy()
hi = plot_stats.bootstrap_95_ci_high.to_numpy()
fig, ax = plt.subplots(figsize=(6.2, 5.0))
ax.axvline(0, color='#555555', linestyle='--', linewidth=1)
ax.errorbar(mid, y, xerr=[mid-lo,hi-mid], fmt='o', capsize=3, color=COLORS['PoolAdapt'])
ax.set(yticks=y, yticklabels=labels, xlabel='Paired mean difference (PoolAdapt − Fixed Prefix)',
       title='Paired quality differences with 95% bootstrap intervals')
ax.grid(axis='x', alpha=.25); ax.invert_yaxis()
fig.tight_layout(); save_figure(fig, 'report_paired_quality_intervals.png')


Saved improvement\improvement_1\figures\report_paired_quality_intervals.png


## 7. Failure analysis

The report's case definitions identify under-allocation candidates, over-allocation candidates, and high-latency observations. These are descriptive criteria, not causal categories. Query-level examples in the original figure are not repeated because the saved figure normalized criterion magnitude in a way that obscured raw differences.

In [9]:
counts = failures.failure_category.value_counts()
order = ['under-allocation candidate','over-allocation candidate','high-latency case']
counts = counts.reindex(order, fill_value=0)
fig, ax = plt.subplots(figsize=(6.2, 3.8))
bars = ax.bar(['Under-allocation','Over-allocation','High latency'], counts.values,
              color=['#b84747','#d47a12','#555555'])
for bar, value in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, value+max(counts.values)*.02, str(value), ha='center', va='bottom')
ax.set(ylabel='Flagged records', title='Descriptive failure-analysis records')
ax.set_ylim(0,max(counts.values)*1.15); ax.grid(axis='y',alpha=.2)
fig.tight_layout(); save_figure(fig, 'report_failure_case_counts.png')


Saved improvement\improvement_1\figures\report_failure_case_counts.png


## 8. Query-group analysis

Group assignments are outcome-independent descriptive median splits. The BM25–Dense dimension is Top-20 candidate-set Jaccard (overlap), not rank-order agreement. Its component rankings were reconstructed; the exact RRF match validates fused ordering only.

In [10]:
def group_metric(strategy: str, budget: int, metric: str) -> pd.DataFrame:
    return groups[(groups.grouping_strategy == strategy) &
                  (groups.target_budget.astype(int) == budget) &
                  (groups.metric == metric)].copy()

strategies = [('redundancy_group','Lexical redundancy'),
              ('sparse_dense_agreement_group','BM25–Dense Top-20 set overlap')]
fig, axes = plt.subplots(1,2,figsize=(6.2,4.1),sharey=True)
for ax,(strategy,title) in zip(axes,strategies):
    part = group_metric(strategy,30,'nDCG@10')
    order = ['redundancy_low','redundancy_high'] if strategy == 'redundancy_group' else ['low_agreement','high_agreement']
    part['_order'] = part.group_name.map({name:index for index,name in enumerate(order)})
    part = part.sort_values('_order')
    names = part.group_name.astype(str).tolist()
    x = np.arange(len(part)); width=.34
    ax.bar(x-width/2,part.Fixed_Prefix_mean,width,label='Fixed Prefix',color=COLORS['Fixed Prefix'])
    ax.bar(x+width/2,part.PoolAdapt_mean,width,label='PoolAdapt',color=COLORS['PoolAdapt'])
    ax.set(xticks=x,xticklabels=[n.replace('redundancy_','').replace('agreement_','') for n in names],
           title=title,ylim=(0,1),ylabel='Mean nDCG@10' if ax is axes[0] else None)
    ax.grid(axis='y',alpha=.2)
axes[-1].legend(frameon=False)
fig.suptitle('Descriptive group outcomes at target B30',y=1.02)
fig.tight_layout(); save_figure(fig, 'report_query_group_b30_quality.png')

fig, axes = plt.subplots(1,2,figsize=(6.2,3.8))
for ax,(strategy,title) in zip(axes,strategies):
    sizes = groups[(groups.grouping_strategy == strategy) & (groups.metric == 'nDCG@10') & (groups.target_budget.astype(int) == 30)]
    sizes = sizes.drop_duplicates('group_name')
    ax.bar(sizes.group_name.str.replace('redundancy_','').str.replace('agreement_',''),
           sizes.query_count, color=['#7b9db7','#2463a6'])
    ax.set(title=title, ylabel='TEST queries')
    ax.grid(axis='y',alpha=.2)
fig.suptitle('Group size at the descriptive median split',y=1.02)
fig.tight_layout(); save_figure(fig, 'report_query_group_sizes.png')


Saved improvement\improvement_1\figures\report_query_group_b30_quality.png
Saved improvement\improvement_1\figures\report_query_group_sizes.png


## 9. Random Percentile Aggregation

The repository inventory contains no Random Percentile Aggregation implementation or corresponding experiment configuration/results. The `random_latency_seed_percentiles.csv` and `random_latency_percentile_sensitivity.csv` files describe latency percentiles for the Random-selection baseline; they are not aggregation-method experiments. The notebook therefore reports this audit finding and creates no synthetic comparison figure.

In [11]:
print('Skipped: no Random Percentile Aggregation implementation/configuration/results were found in the audited Improvement 1 artifacts.')
print('Random-baseline latency percentiles are a separate latency analysis and are not treated as an aggregation-method result.')


Skipped: no Random Percentile Aggregation implementation/configuration/results were found in the audited Improvement 1 artifacts.
Random-baseline latency percentiles are a separate latency analysis and are not treated as an aggregation-method result.


## 10. Export and summary

The cell below lists every figure created by this notebook and checks that each file exists and is non-empty. Existing figures not generated here remain untouched.

In [12]:
missing_outputs = [p for p in generated if not p.is_file() or p.stat().st_size == 0]
if missing_outputs:
    raise RuntimeError('Figure export failed: ' + ', '.join(str(p) for p in missing_outputs))
print(f'Generated {len(generated)} report figures:')
for path in generated:
    print(f'  {path.name}: {path.stat().st_size:,} bytes')


Generated 8 report figures:
  report_matched_budget_quality.png: 98,493 bytes
  report_actual_reranked_pairs.png: 101,118 bytes
  report_reranking_latency.png: 49,625 bytes
  report_quality_cost_pareto.png: 86,507 bytes
  report_paired_quality_intervals.png: 80,199 bytes
  report_failure_case_counts.png: 42,484 bytes
  report_query_group_b30_quality.png: 56,848 bytes
  report_query_group_sizes.png: 57,745 bytes
